# Overwrite a Delta table, then use VERSION AS OF to recover the pre-overwrite snapshot before close of business

Friday at 2pm, the finance lead pasted the wrong CSV into a notebook cell and ran `INSERT OVERWRITE` on `quarterly_numbers`. The 1pm version is the one the auditor wanted. Snowflake costs hundreds per zero-copy clone to do this; Postgres needs a PITR replay; on Delta Lake the answer is one SELECT statement and the cost is zero. You have one session to prove it works, then write a one-paragraph runbook the on-call engineer can use without you next time.

The hands-on work:

- Create a managed Delta table `workspace.default.labrun_time_travel.quarterly_numbers` with five clean rows.
- Run an `UPDATE` that corrects one row (the auditor wanted that one), then a destructive `INSERT OVERWRITE` (the half-loaded CSV simulation) that wipes the original rows.
- Inspect the commit history with `DESCRIBE HISTORY` and identify the version immediately before the overwrite.
- Recover the pre-overwrite snapshot with both `VERSION AS OF N` and `TIMESTAMP AS OF '...'` and write the recovery into a sibling table.

**Time.** Plan for about 50 minutes hands-on. Reading `DESCRIBE HISTORY` and picking the right version is the wall-clock part. Budget up to 80 minutes for the session window.

**Cost.** Zero dollars. Delta time travel is a metadata read, not a data movement. The five-row table writes a few KB of Parquet and a handful of `_delta_log` JSON commits; Free Edition swallows it.

This lab maps to Databricks DE Associate Domain 1 (Platform / Delta Lake) and Domain 3 (Transformation), provisional weight rollup ~10% + 31%.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers use snake_case under the
# labrun_ prefix because Unity Catalog identifiers prefer snake_case.

import atexit
import getpass
import json
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "delta-lake-crud-and-time-travel"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_time_travel"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
QUARTERLY_TABLE_FQN = f"{LAB_SCHEMA_FQN}.quarterly_numbers"
RECOVERED_TABLE_FQN = f"{LAB_SCHEMA_FQN}.quarterly_numbers_recovered"

# Deterministic five-row baseline. Decimal values are spelled out as numeric
# literals so the SELECT-back round-trip on the validator is stable.
BASELINE_ROWS = [
    ("2025-Q1", 1200000.00),
    ("2025-Q2", 1350000.00),
    ("2025-Q3", 1410000.00),
    ("2025-Q4", 1525000.00),
    ("2026-Q1", 1640000.00),
]

# The destructive overwrite. Three rows; deliberate NULL revenue on Q3.
OVERWRITE_ROWS = [
    ("2026-Q1", 1640000.00),
    ("2026-Q2", 1700000.00),
    ("2026-Q3", None),
]

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse, and expose a run_sql() helper. Mirrors Lab 1.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Recreate the Starter warehouse and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. Manifest order:
# recovered table first (created last in Task 4), then the original
# quarterly_numbers table, then the schema with CASCADE. The CASCADE drop
# also cleans up the `_delta_log` history; the lab does NOT run VACUUM, so
# nothing is recovered after cleanup runs.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=RECOVERED_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {RECOVERED_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=QUARTERLY_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {QUARTERLY_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume":
            catalog, schema, volume = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.volumes "
                f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                f"AND volume_name = '{volume}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                return len(list(w.files.list_directory_contents(volume_path))) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
            (
                "SELECT catalog_name, schema_name, volume_name FROM system.information_schema.volume_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, or drop")
    print("them manually with the SQL commands shown by the cleanup cell.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up the schema and the five-row baseline

The lab needs a deterministic seed so the validator can assert on exact values. Build:

1. `CREATE SCHEMA workspace.default.labrun_time_travel` and tag it with `labrun_lab_slug`.
2. `CREATE TABLE workspace.default.labrun_time_travel.quarterly_numbers (quarter STRING, revenue_usd DECIMAL(12,2), loaded_at TIMESTAMP) USING DELTA` and tag the table.
3. Insert all five rows in a single `INSERT INTO ... VALUES (...)` statement. Use `current_timestamp()` for `loaded_at` (every row gets the same near-identical timestamp; that is fine).

The five rows: `2025-Q1 1_200_000.00`, `2025-Q2 1_350_000.00`, `2025-Q3 1_410_000.00`, `2025-Q4 1_525_000.00`, `2026-Q1 1_640_000.00`. The validator reads them back ordered by `quarter`.

In [ ]:
# NBVAL_SKIP
# Task 1: Schema, table, five-row baseline. Tag both the schema and the
# table so the orphan scan can find them.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Run ALTER SCHEMA LAB_SCHEMA_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS QUARTERLY_TABLE_FQN with three
# columns (quarter STRING, revenue_usd DECIMAL(12,2), loaded_at TIMESTAMP)
# USING DELTA via run_sql().

# YOUR CODE: Run ALTER TABLE QUARTERLY_TABLE_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# YOUR CODE: Insert the five baseline rows in a single INSERT INTO ... VALUES
# statement. Use current_timestamp() for loaded_at. The five (quarter,
# revenue_usd) pairs are in BASELINE_ROWS; you can either inline them or
# build a VALUES list from the list.

print(f"Baseline loaded into {QUARTERLY_TABLE_FQN}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Schema tagged, table tagged, five rows present with the
# exact (quarter, revenue_usd) values from BASELINE_ROWS.


def checkpoint_1(session):
    try:
        schema_sql = (
            "SELECT 1 FROM system.information_schema.schemata "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}'"
        )
        if not run_sql(schema_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Schema {LAB_SCHEMA_FQN} not found.",
            )

        schema_tag_sql = (
            "SELECT tag_value FROM system.information_schema.schema_tags "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}' "
            f"AND tag_name = '{LAB_TAG_KEY}'"
        )
        tag_rows = run_sql(schema_tag_sql)["rows"]
        if not any(r[0] == LAB_TAG_VALUE for r in tag_rows):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schema {LAB_SCHEMA_FQN} is missing tag "
                    f"{LAB_TAG_KEY}={LAB_TAG_VALUE}. Apply ALTER SCHEMA SET TAGS."
                ),
            )

        table_sql = (
            "SELECT table_type, data_source_format "
            "FROM system.information_schema.tables "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = 'quarterly_numbers'"
        )
        table_rows = run_sql(table_sql)["rows"]
        if not table_rows:
            return CheckpointResult(
                status="fail",
                error_reason=f"Table {QUARTERLY_TABLE_FQN} not found.",
            )
        table_type, fmt = table_rows[0]
        if table_type != "MANAGED" or (fmt or "").upper() != "DELTA":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {QUARTERLY_TABLE_FQN} is {table_type}/{fmt}; "
                    f"expected MANAGED/DELTA."
                ),
            )

        count_rows = run_sql(f"SELECT count(*) FROM {QUARTERLY_TABLE_FQN}")["rows"]
        row_count = int(count_rows[0][0]) if count_rows else 0
        if row_count != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{QUARTERLY_TABLE_FQN} has {row_count} rows; expected 5. "
                    f"Truncate and re-insert the BASELINE_ROWS."
                ),
            )

        values_rows = run_sql(
            f"SELECT quarter, CAST(revenue_usd AS DOUBLE) "
            f"FROM {QUARTERLY_TABLE_FQN} ORDER BY quarter"
        )["rows"]
        actual = [(str(r[0]), float(r[1])) for r in values_rows]
        expected = list(BASELINE_ROWS)
        if actual != expected:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Row values do not match. Expected {expected}; got {actual}. "
                    f"Re-insert with the exact (quarter, revenue_usd) pairs from "
                    f"BASELINE_ROWS."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message says which piece is missing: the schema, the schema tag, the table, the table format, the row count, or the row values. Each shape is a different SQL statement to run.

</details>

<details><summary>Hint 2 (direction)</summary>

Five statements: `CREATE SCHEMA`, `ALTER SCHEMA ... SET TAGS`, `CREATE TABLE`, `ALTER TABLE ... SET TAGS`, `INSERT INTO ... VALUES`. The INSERT can be a single multi-row statement. For revenue values, use unsigned decimal literals (e.g., `1200000.00`); Spark SQL accepts numeric underscores in some versions but plain digits are universally safe.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
run_sql(
    f"CREATE TABLE IF NOT EXISTS {QUARTERLY_TABLE_FQN} ("
    f"  quarter STRING, revenue_usd DECIMAL(12,2), loaded_at TIMESTAMP"
    f") USING DELTA"
)
run_sql(f"ALTER TABLE {QUARTERLY_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

values_clauses = [
    f"('{q}', {rev}, current_timestamp())" for (q, rev) in BASELINE_ROWS
]
run_sql(f"INSERT INTO {QUARTERLY_TABLE_FQN} VALUES " + ", ".join(values_clauses))
```

</details>

**Wallet check.** Still at $0.00. Five rows, one tiny Parquet file, three `_delta_log` JSON commits. Free Edition has barely noticed.

## Task 2: Run UPDATE and the destructive INSERT OVERWRITE

Two operations, in order:

1. The auditor wanted the 2025-Q4 revenue corrected upward to `1530000.00`. Run `UPDATE quarterly_numbers SET revenue_usd = 1530000.00 WHERE quarter = '2025-Q4'`.

2. The finance lead's mistake: an `INSERT OVERWRITE quarterly_numbers VALUES ('2026-Q1', 1640000.00, current_timestamp()), ('2026-Q2', 1700000.00, current_timestamp()), ('2026-Q3', NULL, current_timestamp())`. This wipes all five existing rows and replaces them with three new rows, one of which has a NULL revenue. The half-loaded CSV simulation.

After both statements, `SELECT count(*)` should return 3 and the 2025-Q4 row should no longer exist in the current state. The commit history (next task) will show every step.

In [ ]:
# NBVAL_SKIP
# Task 2: UPDATE then INSERT OVERWRITE. Both target the same table. The
# OVERWRITE wipes the original five rows in a single atomic commit.

# YOUR CODE: Run UPDATE QUARTERLY_TABLE_FQN SET revenue_usd = 1530000.00
# WHERE quarter = '2025-Q4' via run_sql().

# YOUR CODE: Run INSERT OVERWRITE QUARTERLY_TABLE_FQN VALUES with the three
# OVERWRITE_ROWS values, using current_timestamp() for loaded_at. The third
# row's revenue_usd is NULL.

count_row = run_sql(f"SELECT count(*) FROM {QUARTERLY_TABLE_FQN}")["rows"]
print(f"Current row count after overwrite: {int(count_row[0][0])}")
print("(Expected: 3.)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: DESCRIBE HISTORY shows the four expected operations in order
# (CREATE TABLE, WRITE for the baseline insert, UPDATE, WRITE-Overwrite); the
# current SELECT count is 3 (post-overwrite); the 2025-Q4 row is no longer
# present.


def checkpoint_2(session):
    try:
        history_result = run_sql(
            f"SELECT version, operation, operationParameters "
            f"FROM (DESCRIBE HISTORY {QUARTERLY_TABLE_FQN}) "
            f"ORDER BY version"
        )
        operations = [str(r[1]) for r in history_result["rows"]]
        if not operations:
            return CheckpointResult(
                status="fail",
                error_reason=f"DESCRIBE HISTORY returned no rows for {QUARTERLY_TABLE_FQN}.",
            )

        has_create = any("CREATE TABLE" in op for op in operations)
        write_count = sum(1 for op in operations if op == "WRITE")
        has_update = any("UPDATE" in op for op in operations)

        if not has_create:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DESCRIBE HISTORY did not show a CREATE TABLE commit. "
                    f"Operations: {operations}"
                ),
            )
        if not has_update:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DESCRIBE HISTORY did not show an UPDATE commit. "
                    f"Did you run the UPDATE on 2025-Q4? Operations: {operations}"
                ),
            )
        if write_count < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DESCRIBE HISTORY shows only {write_count} WRITE commits; "
                    f"expected at least two (baseline INSERT and INSERT OVERWRITE). "
                    f"Operations: {operations}"
                ),
            )

        count_rows = run_sql(f"SELECT count(*) FROM {QUARTERLY_TABLE_FQN}")["rows"]
        row_count = int(count_rows[0][0]) if count_rows else 0
        if row_count > 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Current row count is {row_count}; expected 3 after INSERT "
                    f"OVERWRITE wiped the baseline. Did the overwrite run?"
                ),
            )

        q4_rows = run_sql(
            f"SELECT revenue_usd FROM {QUARTERLY_TABLE_FQN} WHERE quarter = '2025-Q4'"
        )["rows"]
        if q4_rows:
            revenue = q4_rows[0][0]
            if revenue is not None and float(revenue) == 1525000.00:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"2025-Q4 still has the original baseline value 1525000.00. "
                        f"The INSERT OVERWRITE should have wiped this row entirely; "
                        f"it should not appear in the current state."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint inspects `DESCRIBE HISTORY` plus the current state of the table. If the message names a missing operation, you skipped that statement. If the count is wrong, the overwrite landed but with the wrong number of rows.

</details>

<details><summary>Hint 2 (direction)</summary>

Two statements only. The UPDATE: `UPDATE workspace.default.labrun_time_travel.quarterly_numbers SET revenue_usd = 1530000.00 WHERE quarter = '2025-Q4'`. The OVERWRITE: `INSERT OVERWRITE workspace.default.labrun_time_travel.quarterly_numbers VALUES ('2026-Q1', 1640000.00, current_timestamp()), ('2026-Q2', 1700000.00, current_timestamp()), ('2026-Q3', NULL, current_timestamp())`. Three rows. The third has `NULL` revenue.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
run_sql(
    f"UPDATE {QUARTERLY_TABLE_FQN} "
    f"SET revenue_usd = 1530000.00 WHERE quarter = '2025-Q4'"
)

run_sql(
    f"INSERT OVERWRITE {QUARTERLY_TABLE_FQN} VALUES "
    f"('2026-Q1', 1640000.00, current_timestamp()), "
    f"('2026-Q2', 1700000.00, current_timestamp()), "
    f"('2026-Q3', NULL, current_timestamp())"
)
```

</details>

**Wallet check.** Still at $0.00. UPDATE rewrote one tiny Parquet file; INSERT OVERWRITE rewrote another. The `_delta_log` directory now has six small JSON commits. Storage cost is rounding error.

## Task 3: Recover the pre-overwrite snapshot with VERSION AS OF

Two questions to answer with one variable assignment:

1. Read `DESCRIBE HISTORY workspace.default.labrun_time_travel.quarterly_numbers`. Look at the `operation` column. The UPDATE commit is the one you want; it is the version that holds the post-UPDATE, pre-overwrite five rows.

2. Assign that version number to a Python variable named `pre_overwrite_version`. The checkpoint runs `SELECT count(*) FROM ... VERSION AS OF pre_overwrite_version` and expects 5; it also checks that 2025-Q4 revenue at that version is `1530000.00` (the UPDATE value, NOT the original `1525000.00`).

The exam tests this. The recovery target is the UPDATE-applied snapshot, not the original load.

In [ ]:
# NBVAL_SKIP
# Task 3: Find the UPDATE commit version and verify VERSION AS OF returns
# the post-UPDATE, pre-overwrite five rows.

history = run_sql(
    f"SELECT version, operation FROM (DESCRIBE HISTORY {QUARTERLY_TABLE_FQN}) "
    f"ORDER BY version"
)
for row in history["rows"]:
    print(f"version={row[0]}  operation={row[1]}")

# YOUR CODE: Read history["rows"] and assign the version of the UPDATE
# commit to pre_overwrite_version (as an int). The UPDATE commit is the
# row whose operation column equals 'UPDATE'.

print(f"pre_overwrite_version = {pre_overwrite_version}")

# Sanity check from the notebook: count rows at that version.
sanity = run_sql(
    f"SELECT count(*) FROM {QUARTERLY_TABLE_FQN} VERSION AS OF {pre_overwrite_version}"
)
print(f"Row count at version {pre_overwrite_version}: {int(sanity['rows'][0][0])}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: The pre_overwrite_version variable resolves to the UPDATE
# commit; VERSION AS OF returns 5 rows; the 2025-Q4 revenue at that version
# is 1530000.00 (the UPDATE value, proving the recovery target is
# post-UPDATE, not the original load).


def checkpoint_3(session):
    try:
        try:
            version = int(pre_overwrite_version)
        except NameError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Variable pre_overwrite_version is not defined. "
                    "Read DESCRIBE HISTORY, find the UPDATE row, and assign "
                    "its version column to pre_overwrite_version."
                ),
            )
        except (TypeError, ValueError):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"pre_overwrite_version is not an int: {pre_overwrite_version!r}. "
                    f"Assign the integer version of the UPDATE commit."
                ),
            )

        count_rows = run_sql(
            f"SELECT count(*) FROM {QUARTERLY_TABLE_FQN} VERSION AS OF {version}"
        )["rows"]
        row_count = int(count_rows[0][0]) if count_rows else 0
        if row_count != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"VERSION AS OF {version} returned {row_count} rows; expected 5. "
                    f"This is either the wrong version (the OVERWRITE commit has 3 rows) "
                    f"or before the baseline INSERT (0 rows). Pick the UPDATE commit version."
                ),
            )

        q4_rows = run_sql(
            f"SELECT revenue_usd FROM {QUARTERLY_TABLE_FQN} "
            f"VERSION AS OF {version} WHERE quarter = '2025-Q4'"
        )["rows"]
        if not q4_rows:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"VERSION AS OF {version} has no 2025-Q4 row; you may have "
                    f"picked a version before the baseline INSERT completed."
                ),
            )
        revenue = q4_rows[0][0]
        if revenue is None or float(revenue) != 1530000.00:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"At VERSION AS OF {version}, 2025-Q4 revenue_usd is {revenue!r}; "
                    f"expected 1530000.00 (the UPDATE value). If you picked the "
                    f"baseline INSERT version you would see 1525000.00; pick the "
                    f"UPDATE version instead."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message tells you exactly which version you picked. If 2025-Q4 revenue is `1525000.00` you picked the version BEFORE the UPDATE landed (the baseline INSERT). If row count is 3 you picked the OVERWRITE version itself. The target is the UPDATE row.

</details>

<details><summary>Hint 2 (direction)</summary>

`DESCRIBE HISTORY` returns one row per commit, ordered chronologically. The `operation` column has values like `CREATE TABLE`, `WRITE`, `UPDATE`. Find the one whose operation is exactly `UPDATE` and pull its `version` value. Assign that integer to `pre_overwrite_version`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
history = run_sql(
    f"SELECT version, operation FROM (DESCRIBE HISTORY {QUARTERLY_TABLE_FQN}) "
    f"ORDER BY version"
)
pre_overwrite_version = None
for row in history["rows"]:
    if str(row[1]) == "UPDATE":
        pre_overwrite_version = int(row[0])
        break

if pre_overwrite_version is None:
    raise RuntimeError("No UPDATE commit found; re-run Task 2.")
```

</details>

**Wallet check.** Still at $0.00. Three more SELECTs against tiny snapshots. The expensive part of this task was your wall-clock time reading `DESCRIBE HISTORY` and matching the operation column. The Delta engine did the recovery in microseconds.

## Task 4: Recover via TIMESTAMP AS OF and materialize the snapshot

Two final pieces:

1. Capture the timestamp of the UPDATE commit from `DESCRIBE HISTORY`. Run `SELECT count(*) FROM workspace.default.labrun_time_travel.quarterly_numbers TIMESTAMP AS OF '<captured_timestamp>'` and confirm it returns 5. The timestamp lookup and the version lookup should both resolve to the same snapshot.

2. Materialize the recovery: `CREATE OR REPLACE TABLE workspace.default.labrun_time_travel.quarterly_numbers_recovered AS SELECT * FROM workspace.default.labrun_time_travel.quarterly_numbers VERSION AS OF pre_overwrite_version`. This writes a sibling managed table that holds the recovered five rows. Tag it.

The auditor gets the recovered table, not a query result they have to re-run. Time travel is the recovery primitive; CREATE OR REPLACE is the operational handoff.

In [ ]:
# NBVAL_SKIP
# Task 4: TIMESTAMP AS OF recovery and CREATE OR REPLACE materialization.

# Find the timestamp of the UPDATE commit.
history_ts = run_sql(
    f"SELECT version, operation, timestamp "
    f"FROM (DESCRIBE HISTORY {QUARTERLY_TABLE_FQN}) ORDER BY version"
)
update_timestamp = None
for row in history_ts["rows"]:
    if str(row[1]) == "UPDATE":
        update_timestamp = str(row[2])
        break

if update_timestamp is None:
    raise RuntimeError("No UPDATE commit found; re-run Task 2 first.")

print(f"UPDATE commit timestamp: {update_timestamp}")

# YOUR CODE: Run SELECT count(*) FROM QUARTERLY_TABLE_FQN TIMESTAMP AS OF
# '<update_timestamp>' via run_sql() and print the row count. The expected
# answer is 5.

# YOUR CODE: Materialize the recovery with CREATE OR REPLACE TABLE
# RECOVERED_TABLE_FQN AS SELECT * FROM QUARTERLY_TABLE_FQN VERSION AS OF
# pre_overwrite_version via run_sql().

# YOUR CODE: Tag the recovered table with ALTER TABLE RECOVERED_TABLE_FQN
# SET TAGS ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

print(f"Recovered table written: {RECOVERED_TABLE_FQN}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: TIMESTAMP AS OF on the UPDATE commit returns 5 rows; a
# materialized recovered table exists with 5 rows; the recovered table is
# tagged.


def checkpoint_4(session):
    try:
        history = run_sql(
            f"SELECT version, operation, timestamp "
            f"FROM (DESCRIBE HISTORY {QUARTERLY_TABLE_FQN}) ORDER BY version"
        )
        ts = None
        for row in history["rows"]:
            if str(row[1]) == "UPDATE":
                ts = str(row[2])
                break
        if ts is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "DESCRIBE HISTORY does not contain an UPDATE commit. "
                    "Re-run Task 2 before this checkpoint."
                ),
            )

        ts_count_rows = run_sql(
            f"SELECT count(*) FROM {QUARTERLY_TABLE_FQN} "
            f"TIMESTAMP AS OF '{ts}'"
        )["rows"]
        ts_row_count = int(ts_count_rows[0][0]) if ts_count_rows else 0
        if ts_row_count != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"TIMESTAMP AS OF '{ts}' returned {ts_row_count} rows; "
                    f"expected 5. The timestamp lookup should resolve to the "
                    f"same snapshot as VERSION AS OF."
                ),
            )

        recovered_sql = (
            "SELECT 1 FROM system.information_schema.tables "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = 'quarterly_numbers_recovered'"
        )
        if not run_sql(recovered_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {RECOVERED_TABLE_FQN} not found. Run CREATE OR REPLACE "
                    f"TABLE ... AS SELECT * FROM {QUARTERLY_TABLE_FQN} VERSION AS OF "
                    f"pre_overwrite_version."
                ),
            )

        recovered_count_rows = run_sql(
            f"SELECT count(*) FROM {RECOVERED_TABLE_FQN}"
        )["rows"]
        recovered_count = int(recovered_count_rows[0][0]) if recovered_count_rows else 0
        if recovered_count != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{RECOVERED_TABLE_FQN} has {recovered_count} rows; expected 5. "
                    f"The CTAS may have read the wrong version."
                ),
            )

        recovered_q4_rows = run_sql(
            f"SELECT revenue_usd FROM {RECOVERED_TABLE_FQN} "
            f"WHERE quarter = '2025-Q4'"
        )["rows"]
        if not recovered_q4_rows or float(recovered_q4_rows[0][0]) != 1530000.00:
            actual = recovered_q4_rows[0][0] if recovered_q4_rows else None
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{RECOVERED_TABLE_FQN} 2025-Q4 revenue is {actual!r}; "
                    f"expected 1530000.00 (the UPDATE value). The CTAS targeted "
                    f"the wrong version."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes. Either the TIMESTAMP AS OF query did not return 5 rows (the timestamp picked the wrong commit) or the recovered table does not match (the CTAS read the wrong version or the table was never created). Read the message before peeking.

</details>

<details><summary>Hint 2 (direction)</summary>

Three statements. (1) `SELECT count(*) FROM ... TIMESTAMP AS OF '<update_timestamp>'` to prove the timestamp lookup works. (2) `CREATE OR REPLACE TABLE workspace.default.labrun_time_travel.quarterly_numbers_recovered AS SELECT * FROM workspace.default.labrun_time_travel.quarterly_numbers VERSION AS OF pre_overwrite_version`. (3) `ALTER TABLE ... SET TAGS ('labrun_lab_slug' = '...')` on the recovered table.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
ts_count = run_sql(
    f"SELECT count(*) FROM {QUARTERLY_TABLE_FQN} "
    f"TIMESTAMP AS OF '{update_timestamp}'"
)
print(f"TIMESTAMP AS OF count: {int(ts_count['rows'][0][0])}")

run_sql(
    f"CREATE OR REPLACE TABLE {RECOVERED_TABLE_FQN} AS "
    f"SELECT * FROM {QUARTERLY_TABLE_FQN} "
    f"VERSION AS OF {pre_overwrite_version}"
)
run_sql(f"ALTER TABLE {RECOVERED_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
```

</details>

**Wallet check.** Still at $0.00. The CTAS copies five rows; the resulting Parquet is microscopic. The auditor's table is now durable and ready for handoff.

## Cleanup

Time to tear it all down. The cell below drops both managed tables (recovered first, then the original), then drops the lab schema with `CASCADE`. The `_delta_log` history goes away with the schema. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_destroyed = 0
standard_destroyed = len(CLEANUP_MANIFEST) - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** Delta time travel is a metadata read. Snowflake calls this Time Travel and bills per-cloned-byte; on Databricks Free Edition it is free. The expensive part of this session was your wall-clock time reading commit history. The recovered table is gone, the original table is gone, the schema is gone, and the daily compute quota is the only thing this session touched.

## Reflection

These are not graded. They are for you.

1. Walk through what Delta Lake does when you run `INSERT OVERWRITE` on a managed table. Name each thing that happens at the storage layer (file writes, file deletes, `_delta_log` JSON commit, version number increment). Why does `VERSION AS OF` remain possible after the overwrite even though the "old data" was logically wiped?

2. Your team wants to run `VACUUM` on the table every Monday morning to reclaim storage from old files. What is the operational risk of using the default retention duration vs a 1-hour custom retention? Sketch the SQL command you would run and the trade-off you accept.

## Exam-Style Practice

**Question 1 (Domain 1, Delta Lake commit log):**

A data engineer runs the following sequence on a managed Delta table `sales.orders`:

```sql
INSERT INTO sales.orders VALUES (1, 'A'), (2, 'B');
UPDATE sales.orders SET col2 = 'C' WHERE col1 = 1;
INSERT OVERWRITE sales.orders VALUES (10, 'X');
```

After the sequence, `DESCRIBE HISTORY sales.orders` shows versions 0 (CREATE), 1 (WRITE), 2 (UPDATE), 3 (WRITE-Overwrite). The engineer wants the row set as it existed immediately after the UPDATE but before the OVERWRITE. Which statement returns that row set?

A. `SELECT * FROM sales.orders VERSION AS OF 1`

B. `SELECT * FROM sales.orders VERSION AS OF 2`

C. `SELECT * FROM sales.orders WHERE _commit_version = 2`

D. `RESTORE TABLE sales.orders TO VERSION AS OF 2` followed by `SELECT * FROM sales.orders`

<details><summary>Show answer</summary>

**Correct: B.**

- A returns the row set after the initial INSERT but before the UPDATE (col1=1 still has col2='B'). It is the wrong snapshot for the auditor's ask.
- B is correct: VERSION AS OF 2 returns the row set as committed by version 2, which is the UPDATE commit. The 2-row table has col1=1, col2='C' and col1=2, col2='B'.
- C is wrong: there is no `_commit_version` filter syntax in Delta Lake; commit version is a snapshot selector, not a per-row column.
- D works mechanically but mutates the table state. The auditor only wants to READ the pre-overwrite snapshot; `RESTORE` rewrites the table to that version and clobbers the most recent overwrite, which is destructive and is rarely the right answer for a read-only audit.

</details>

**Question 2 (Domain 1, time travel retention):**

A platform engineer wants to free up storage on a Delta table that has 90 days of commit history. They run `VACUUM sales.orders RETAIN 1 HOURS`. Later, an analyst runs `SELECT * FROM sales.orders VERSION AS OF 50` (a version older than 1 hour) and gets a `FileNotFoundException`. Why?

A. VACUUM with `RETAIN 1 HOURS` deleted the underlying Parquet files that VERSION AS OF 50 references; the commit log entry still exists but the files are gone.

B. The Delta protocol caps time travel at 100 commits; VERSION AS OF 50 falls outside the cap.

C. The `_delta_log` directory is corrupted; `MSCK REPAIR TABLE` will restore the history.

D. VACUUM rewrites the `_delta_log` JSON and removes commit entries older than the retention window.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: VACUUM physically deletes Parquet files older than the retention duration. The commit log entry still references them, so `VERSION AS OF` can find the metadata, but reading the snapshot fails because the underlying files are gone. The default `delta.deletedFileRetentionDuration` is 7 days for exactly this reason; the engineer overrode it to 1 hour and broke their own time travel.
- B is wrong: Delta has no 100-commit cap. Retention is duration-based, not count-based.
- C is wrong: `MSCK REPAIR TABLE` is a Hive command for partition discovery and does not apply to Delta tables. The `_delta_log` is not corrupted in this scenario.
- D is wrong: VACUUM removes data files but does not rewrite the `_delta_log` JSON entries. The commit metadata persists; the data files do not.

</details>